<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-04-image-and-multimodal-generation/lesson-4.1-diffusion-models/notebooks/exercises-4.1.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 4.1 — Diffusion Models: From Noise to Art
Netsetos GenAI Course v2.0 | Module 4

Covers: DDPM math, latent diffusion, text conditioning, practical generation with diffusers


In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
# pip install torch torchvision diffusers transformers -q


## Ex 1: Forward Process — Add Noise


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Create a simple 8x8 'image'
image = np.zeros((8, 8))
image[2:6, 2:6] = 1.0  # White square

# Linear noise schedule
T = 1000
beta = np.linspace(0.0001, 0.02, T)
alpha = 1 - beta
alpha_bar = np.cumprod(alpha)

# Forward process: q(x_t | x_0) = N(√ᾱ_t · x_0, (1-ᾱ_t)·I)
def forward(x_0, t):
    noise = np.random.randn(*x_0.shape)
    x_t = np.sqrt(alpha_bar[t]) * x_0 + np.sqrt(1 - alpha_bar[t]) * noise
    return x_t, noise

fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for i, t in enumerate([0, 100, 300, 500, 700, 999]):
    x_t, _ = forward(image, t)
    axes[i].imshow(x_t, cmap='gray', vmin=-2, vmax=2)
    axes[i].set_title(f't={t}')
    axes[i].axis('off')
plt.suptitle('Forward Process: Image → Noise')
plt.tight_layout()
plt.show()


## Ex 2: Cosine vs Linear Schedule


In [ ]:
def cosine_schedule(T, s=0.008):
    steps = np.arange(T + 1)
    f = np.cos((steps / T + s) / (1 + s) * np.pi / 2) ** 2
    alpha_bar = f / f[0]
    beta = 1 - alpha_bar[1:] / alpha_bar[:-1]
    return np.clip(beta, 0, 0.999)

beta_linear = np.linspace(0.0001, 0.02, T)
beta_cosine = cosine_schedule(T)

alpha_bar_lin = np.cumprod(1 - beta_linear)
alpha_bar_cos = np.cumprod(1 - beta_cosine)

plt.figure(figsize=(10, 4))
plt.plot(alpha_bar_lin, label='Linear', linewidth=2)
plt.plot(alpha_bar_cos, label='Cosine', linewidth=2)
plt.xlabel('Timestep t')
plt.ylabel('ᾱ_t (signal remaining)')
plt.title('Signal Preservation: Cosine preserves details longer')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Ex 3: Tiny Denoising Network


In [ ]:
import torch
import torch.nn as nn

class TinyDenoiser(nn.Module):
    def __init__(self, img_size=8):
        super().__init__()
        dim = img_size * img_size
        self.net = nn.Sequential(
            nn.Linear(dim + 1, 256),  # +1 for timestep
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, dim),
        )
    
    def forward(self, x_t, t):
        # x_t: (batch, 64), t: (batch, 1)
        x = torch.cat([x_t, t], dim=-1)
        return self.net(x)  # Predict noise ε

model = TinyDenoiser()
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

# Test forward
x_t = torch.randn(4, 64)  # Batch of 4 noisy 8×8 images
t = torch.rand(4, 1)      # Random timesteps
noise_pred = model(x_t, t)
print(f'Input: {x_t.shape}, Predicted noise: {noise_pred.shape}')


## Ex 4: Training Loop


In [ ]:
# Training data: 8x8 images with white squares at random positions
def make_dataset(n=1000, size=8):
    images = []
    for _ in range(n):
        img = np.zeros((size, size))
        x, y = np.random.randint(0, size-3), np.random.randint(0, size-3)
        img[x:x+3, y:y+3] = 1.0
        images.append(img.flatten())
    return torch.tensor(np.array(images), dtype=torch.float32)

dataset = make_dataset(1000)
print(f'Dataset: {dataset.shape}')

# Training loop
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
beta_t = torch.linspace(0.0001, 0.02, T)
alpha_t = 1 - beta_t
alpha_bar_t = torch.cumprod(alpha_t, dim=0)

for epoch in range(10):
    total_loss = 0
    for i in range(0, len(dataset), 32):
        batch = dataset[i:i+32]
        t = torch.randint(0, T, (len(batch),))
        noise = torch.randn_like(batch)
        
        # Forward process
        ab = alpha_bar_t[t].unsqueeze(1)
        x_t = torch.sqrt(ab) * batch + torch.sqrt(1 - ab) * noise
        
        # Predict noise
        t_norm = t.float().unsqueeze(1) / T
        noise_pred = model(x_t, t_norm)
        
        loss = nn.MSELoss()(noise_pred, noise)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    if epoch % 2 == 0:
        print(f'Epoch {epoch}: loss={total_loss/(len(dataset)//32):.4f}')


## Ex 5: Reverse Process — Generate!


In [ ]:
@torch.no_grad()
def sample(model, T=1000, n=8, img_size=8):
    x = torch.randn(n, img_size*img_size)
    for t in reversed(range(T)):
        t_tensor = torch.full((n, 1), t / T)
        noise_pred = model(x, t_tensor)
        
        alpha = alpha_t[t]
        alpha_b = alpha_bar_t[t]
        
        x = (1/torch.sqrt(alpha)) * (x - (beta_t[t]/torch.sqrt(1-alpha_b)) * noise_pred)
        if t > 0:
            x += torch.sqrt(beta_t[t]) * torch.randn_like(x)
    return x.reshape(n, img_size, img_size).numpy()

samples = sample(model)
fig, axes = plt.subplots(1, 8, figsize=(16, 2))
for i in range(8):
    axes[i].imshow(samples[i], cmap='gray')
    axes[i].axis('off')
plt.suptitle('Generated Images (from pure noise!)')
plt.tight_layout()
plt.show()


## Ex 6: CFG Scale Experiment


In [ ]:
# Simulating CFG effect (conceptual — real implementation needs a text encoder)
def simulate_cfg(cond_noise, uncond_noise, scale):
    return uncond_noise + scale * (cond_noise - uncond_noise)

cond = np.array([0.8, -0.3, 0.5])   # Noise prediction WITH prompt
uncond = np.array([0.1, 0.2, -0.1]) # Noise prediction WITHOUT prompt

for scale in [1.0, 3.0, 7.5, 12.0, 20.0]:
    guided = simulate_cfg(cond, uncond, scale)
    magnitude = np.linalg.norm(guided)
    print(f'CFG={scale:5.1f}: {np.round(guided, 2)} | magnitude={magnitude:.2f}')

print('\nScale 1.0: ignores prompt (uncond only)')
print('Scale 7.5: follows prompt well')
print('Scale 20.0: oversaturated/artifacts')


## Ex 7: Generate with diffusers (GPU required)


In [ ]:
# from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler
# import torch
# 
# pipe = StableDiffusionXLPipeline.from_pretrained(
#     'stabilityai/stable-diffusion-xl-base-1.0',
#     torch_dtype=torch.float16, variant='fp16'
# ).to('cuda')
# pipe.scheduler = DPMSolverMultistepScheduler.from_config(
#     pipe.scheduler.config, algorithm_type='dpmsolver++', use_karras_sigmas=True
# )
# 
# image = pipe(
#     prompt='A vibrant Diwali celebration in an Indian village, oil lamps, fireworks, families, warm colors',
#     negative_prompt='blurry, distorted, low quality',
#     num_inference_steps=25, guidance_scale=7.5,
#     width=1024, height=1024,
#     generator=torch.Generator('cuda').manual_seed(42)
# ).images[0]
# image.save('diwali.png')

print('Uncomment above code and run on GPU (Colab T4/A100)')
print('SDXL needs ~7GB VRAM | ~5 seconds per image on A100')


## Ex 8: Evaluation Metrics


In [ ]:
# Simulated evaluation (real metrics need torchmetrics or clean-fid)
def mock_clip_score(prompt, image_embedding):
    '''Cosine similarity between text and image embeddings.'''
    text_emb = np.random.randn(512)  # Mock CLIP text embedding
    cos_sim = np.dot(text_emb, image_embedding) / (
        np.linalg.norm(text_emb) * np.linalg.norm(image_embedding)
    )
    return (cos_sim + 1) / 2  # Scale to 0-1

def mock_aesthetic_score(image_embedding):
    '''Learned aesthetic quality predictor.'''
    return np.random.uniform(4, 8)  # Score out of 10

# Simulate 10 generated images
print(f'{"Prompt":30s} {"CLIP Score":>10s} {"Aesthetic":>10s}')
print('-' * 55)
prompts = [
    'A temple at sunset', 'Abstract noise pattern',
    'Beautiful landscape with mountains', 'Random pixels',
    'Indian wedding celebration',
]
for prompt in prompts:
    emb = np.random.randn(512)
    clip = mock_clip_score(prompt, emb)
    aesthetic = mock_aesthetic_score(emb)
    print(f'{prompt:30s} {clip:>10.3f} {aesthetic:>10.1f}')


## Ex 9: Production Optimization & Tools


In [ ]:
# Production Optimization Checklist
optimizations = {
    'torch.compile': '20-30% speedup, PyTorch 2.0+, one line: pipe.unet = torch.compile(pipe.unet)',
    'float16': '2× less VRAM, minimal quality loss: torch_dtype=torch.float16',
    'TensorRT': '40-60% faster on NVIDIA GPUs, requires tensorrt package',
    'xformers': 'Memory-efficient attention: pipe.enable_xformers_memory_efficient_attention()',
    'VAE slicing': 'Process large batches: pipe.enable_vae_slicing()',
    'Attention slicing': 'Lower VRAM: pipe.enable_attention_slicing()',
}

ui_tools = {
    'ComfyUI': 'Node-based, best for complex pipelines, production workflows',
    'Automatic1111': 'Web UI, most popular, great for exploration',
    'diffusers (Python)': 'Code-first, best for integration into apps/APIs',
}

print('=== Optimization Techniques ===')
for name, desc in optimizations.items():
    print(f'  ⚡ {name}: {desc}')

print('\n=== UI/Workflow Tools ===')
for name, desc in ui_tools.items():
    print(f'  🛠️ {name}: {desc}')


---
## Recap
Diffusion = forward (add noise) + reverse (remove noise). Latent diffusion makes it practical (VAE compresses 48×). Text conditioning via CLIP/T5 + cross-attention. CFG amplifies prompt following. Production: diffusers + DPM++ scheduler + 25 steps. Three 2024-2026 revolutions: DiT, flow matching, consistency models.
